# Mini-Projet: Régularisation des Modèles de Machine Learning

Ce notebook présente différentes techniques de régularisation pour améliorer les performances des modèles de machine learning et éviter le surapprentissage (overfitting).

## Objectifs:
1. Comprendre le problème du surapprentissage
2. Implémenter et comparer différentes techniques de régularisation:
   - Régularisation L1 (Lasso)
   - Régularisation L2 (Ridge)
   - Elastic Net (combinaison L1 + L2)
3. Visualiser et analyser les résultats

## 1. Importation des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuration pour les graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 2. Génération et préparation des données

Nous allons générer un dataset synthétique avec du bruit pour simuler un problème réel.

In [ ]:
# Génération des données
np.random.seed(42)
X, y = make_regression(
    n_samples=200,
    n_features=20,
    n_informative=10,
    noise=20,
    random_state=42
)

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Taille de l'ensemble d'entraînement: {X_train.shape}")
print(f"Taille de l'ensemble de test: {X_test.shape}")
print(f"Nombre de caractéristiques: {X.shape[1]}")

## 3. Modèle de base (sans régularisation)

Commençons par entraîner une régression linéaire simple sans régularisation.

In [ ]:
# Modèle de régression linéaire simple
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Prédictions
y_train_pred_lr = lr.predict(X_train_scaled)
y_test_pred_lr = lr.predict(X_test_scaled)

# Métriques
train_mse_lr = mean_squared_error(y_train, y_train_pred_lr)
test_mse_lr = mean_squared_error(y_test, y_test_pred_lr)
train_r2_lr = r2_score(y_train, y_train_pred_lr)
test_r2_lr = r2_score(y_test, y_test_pred_lr)

print("=" * 50)
print("RÉGRESSION LINÉAIRE (sans régularisation)")
print("=" * 50)
print(f"MSE Train: {train_mse_lr:.2f}")
print(f"MSE Test:  {test_mse_lr:.2f}")
print(f"R² Train:  {train_r2_lr:.4f}")
print(f"R² Test:   {test_r2_lr:.4f}")
print(f"Différence MSE (overfitting): {test_mse_lr - train_mse_lr:.2f}")

## 4. Régularisation L2 (Ridge)

La régularisation Ridge ajoute une pénalité proportionnelle au carré des coefficients.

In [ ]:
# Modèle Ridge avec différentes valeurs de alpha
alphas_ridge = [0.01, 0.1, 1, 10, 100]
ridge_results = []

for alpha in alphas_ridge:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    
    y_train_pred = ridge.predict(X_train_scaled)
    y_test_pred = ridge.predict(X_test_scaled)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    ridge_results.append({
        'alpha': alpha,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'coefficients': ridge.coef_
    })

print("=" * 50)
print("RÉGULARISATION RIDGE (L2)")
print("=" * 50)
for result in ridge_results:
    print(f"\nAlpha: {result['alpha']}")
    print(f"  MSE Train: {result['train_mse']:.2f}")
    print(f"  MSE Test:  {result['test_mse']:.2f}")
    print(f"  R² Test:   {result['test_r2']:.4f}")

## 5. Régularisation L1 (Lasso)

La régularisation Lasso peut réduire certains coefficients à zéro, effectuant ainsi une sélection de caractéristiques.

In [ ]:
# Modèle Lasso avec différentes valeurs de alpha
alphas_lasso = [0.01, 0.1, 1, 10, 100]
lasso_results = []

for alpha in alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    
    y_train_pred = lasso.predict(X_train_scaled)
    y_test_pred = lasso.predict(X_test_scaled)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Nombre de coefficients non nuls
    n_nonzero = np.sum(lasso.coef_ != 0)
    
    lasso_results.append({
        'alpha': alpha,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'coefficients': lasso.coef_,
        'n_nonzero': n_nonzero
    })

print("=" * 50)
print("RÉGULARISATION LASSO (L1)")
print("=" * 50)
for result in lasso_results:
    print(f"\nAlpha: {result['alpha']}")
    print(f"  MSE Train: {result['train_mse']:.2f}")
    print(f"  MSE Test:  {result['test_mse']:.2f}")
    print(f"  R² Test:   {result['test_r2']:.4f}")
    print(f"  Caractéristiques non nulles: {result['n_nonzero']}/{X.shape[1]}")

## 6. Elastic Net (L1 + L2)

Elastic Net combine les régularisations L1 et L2.

In [ ]:
# Modèle Elastic Net
elastic_net = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
elastic_net.fit(X_train_scaled, y_train)

y_train_pred_en = elastic_net.predict(X_train_scaled)
y_test_pred_en = elastic_net.predict(X_test_scaled)

train_mse_en = mean_squared_error(y_train, y_train_pred_en)
test_mse_en = mean_squared_error(y_test, y_test_pred_en)
train_r2_en = r2_score(y_train, y_train_pred_en)
test_r2_en = r2_score(y_test, y_test_pred_en)
n_nonzero_en = np.sum(elastic_net.coef_ != 0)

print("=" * 50)
print("ELASTIC NET (L1 + L2)")
print("=" * 50)
print(f"MSE Train: {train_mse_en:.2f}")
print(f"MSE Test:  {test_mse_en:.2f}")
print(f"R² Train:  {train_r2_en:.4f}")
print(f"R² Test:   {test_r2_en:.4f}")
print(f"Caractéristiques non nulles: {n_nonzero_en}/{X.shape[1]}")

## 7. Visualisations et Comparaisons

In [ ]:
# Comparaison des performances
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Graphique 1: Évolution du MSE en fonction de alpha pour Ridge
alphas = [r['alpha'] for r in ridge_results]
train_mses = [r['train_mse'] for r in ridge_results]
test_mses = [r['test_mse'] for r in ridge_results]

axes[0].plot(alphas, train_mses, 'o-', label='Train MSE', linewidth=2, markersize=8)
axes[0].plot(alphas, test_mses, 's-', label='Test MSE', linewidth=2, markersize=8)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (paramètre de régularisation)', fontsize=12)
axes[0].set_ylabel('MSE', fontsize=12)
axes[0].set_title('Ridge: MSE vs Alpha', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Graphique 2: Évolution du MSE en fonction de alpha pour Lasso
alphas_l = [r['alpha'] for r in lasso_results]
train_mses_l = [r['train_mse'] for r in lasso_results]
test_mses_l = [r['test_mse'] for r in lasso_results]

axes[1].plot(alphas_l, train_mses_l, 'o-', label='Train MSE', linewidth=2, markersize=8)
axes[1].plot(alphas_l, test_mses_l, 's-', label='Test MSE', linewidth=2, markersize=8)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (paramètre de régularisation)', fontsize=12)
axes[1].set_ylabel('MSE', fontsize=12)
axes[1].set_title('Lasso: MSE vs Alpha', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des coefficients
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Régression linéaire
axes[0, 0].bar(range(len(lr.coef_)), lr.coef_, alpha=0.7)
axes[0, 0].set_title('Coefficients - Régression Linéaire', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Index de caractéristique')
axes[0, 0].set_ylabel('Valeur du coefficient')
axes[0, 0].grid(True, alpha=0.3)

# Ridge (alpha=1)
ridge_coef = ridge_results[2]['coefficients']
axes[0, 1].bar(range(len(ridge_coef)), ridge_coef, alpha=0.7, color='orange')
axes[0, 1].set_title('Coefficients - Ridge (α=1)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Index de caractéristique')
axes[0, 1].set_ylabel('Valeur du coefficient')
axes[0, 1].grid(True, alpha=0.3)

# Lasso (alpha=1)
lasso_coef = lasso_results[2]['coefficients']
axes[1, 0].bar(range(len(lasso_coef)), lasso_coef, alpha=0.7, color='green')
axes[1, 0].set_title('Coefficients - Lasso (α=1)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Index de caractéristique')
axes[1, 0].set_ylabel('Valeur du coefficient')
axes[1, 0].grid(True, alpha=0.3)

# Elastic Net
axes[1, 1].bar(range(len(elastic_net.coef_)), elastic_net.coef_, alpha=0.7, color='red')
axes[1, 1].set_title('Coefficients - Elastic Net', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Index de caractéristique')
axes[1, 1].set_ylabel('Valeur du coefficient')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tableau récapitulatif des performances
results_summary = pd.DataFrame({
    'Modèle': ['Linear Regression', 'Ridge (α=1)', 'Lasso (α=1)', 'Elastic Net'],
    'MSE Train': [
        train_mse_lr,
        ridge_results[2]['train_mse'],
        lasso_results[2]['train_mse'],
        train_mse_en
    ],
    'MSE Test': [
        test_mse_lr,
        ridge_results[2]['test_mse'],
        lasso_results[2]['test_mse'],
        test_mse_en
    ],
    'R² Test': [
        test_r2_lr,
        ridge_results[2]['test_r2'],
        lasso_results[2]['test_r2'],
        test_r2_en
    ],
    'Features non nulles': [
        X.shape[1],
        X.shape[1],
        lasso_results[2]['n_nonzero'],
        n_nonzero_en
    ]
})

print("\n" + "=" * 80)
print("RÉSUMÉ COMPARATIF DES MODÈLES")
print("=" * 80)
print(results_summary.to_string(index=False))
print("=" * 80)

In [ ]:
# Visualisation finale: Comparaison des R² scores
fig, ax = plt.subplots(figsize=(12, 6))

models = results_summary['Modèle']
r2_scores = results_summary['R² Test']

bars = ax.bar(models, r2_scores, alpha=0.7, color=['blue', 'orange', 'green', 'red'])
ax.set_ylabel('R² Score (Test)', fontsize=12)
ax.set_title('Comparaison des performances (R² Score sur ensemble de test)', 
             fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Ajouter les valeurs sur les barres
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 8. Conclusions

### Observations principales:

1. **Régression Linéaire**: Modèle de base sans régularisation qui peut souffrir de surapprentissage.

2. **Ridge (L2)**: 
   - Pénalise les coefficients élevés
   - Réduit tous les coefficients mais ne les met pas à zéro
   - Améliore la généralisation

3. **Lasso (L1)**:
   - Peut mettre certains coefficients à zéro
   - Effectue une sélection automatique de caractéristiques
   - Utile pour l'interprétabilité du modèle

4. **Elastic Net**:
   - Combine les avantages de Ridge et Lasso
   - Bon compromis entre sélection de features et réduction des coefficients

### Recommandations:
- Utilisez Ridge quand toutes les features sont importantes
- Utilisez Lasso pour la sélection de features
- Utilisez Elastic Net pour un équilibre entre les deux
- Toujours valider avec la validation croisée pour choisir le meilleur alpha